In [1]:
import os
os.environ["NUMBA_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMBA_DISABLE_JIT"] = "0"

In [2]:
import numpy as np
from numba import njit, prange, float32, uint8
from src.constants import *
from time import time, perf_counter


In [3]:
N = len(NEURON_NAMES)
ALPHA_L = 250
td = np.arange(1, ALPHA_L + 1, dtype=np.float32)
alpha = (td / 30) * np.exp((30 - td) / 30)   

cue_wave = np.zeros(TMAX, dtype=np.float32)
go_wave = np.zeros_like(cue_wave)
cue_wave[EPOCHS['sample'][0]:EPOCHS['sample'][1]] = CUE_STRENGTH
go_wave[EPOCHS['response'][0]:EPOCHS['response'][0] + GO_DURATION] = GO_STRENGTH


# --------------------------------------------------------------------
# Hand‑crafted weights -------------------------------------------------
# --------------------------------------------------------------------
new_jh_weights = [
    ("Somat", "ALMprep", 40),
    ("Somat", "MSN1", 220),
    ("MSN1", "SNR1", -90),
    ("SNR1", "VMprep", -10),
    ("VMprep", "ALMprep", 70),
    ("ALMprep", "VMprep", 80),
    ("ALMprep", "MSN2", 320),
    ("MSN2", "SNR2", -50),
    ("SNR2", "VMresp", -100),
    ("PPN", "THALgo", 60),
    ("THALgo", "ALMinter", 55),
    ("ALMinter", "ALMprep", -50),
    ("THALgo", "ALMresp", 30),
    ("ALMresp", "MSN3", 320),
    ("MSN3", "SNR3", -90),
    ("SNR3", "VMresp", -50),
    ("VMresp", "ALMresp", 85),
    ("ALMresp", "VMresp", 90),
]

# --------------------------------------------------------------------
# Build weight matrix -------------------------------------------------
# --------------------------------------------------------------------
N = len(NEURON_NAMES)
W = np.zeros((N, N), dtype=np.float32)
for pre, post, w in new_jh_weights:
    i = NEURON_NAMES.index(pre)
    j = NEURON_NAMES.index(post)
    W[i, j] += w

pass_ids = [NEURON_NAMES.index(x) for x in ["VMresp", "ALMresp", "SNR3"]]
pass_ids = np.array(pass_ids)
print(pass_ids)

[13 12 11]


In [4]:
# CREATING CRITERION
conditions = []
for condition in CRITERIA:
    condition_criteria = []
    for neuron_name, neuron in CRITERIA[condition].items():
        idx = NEURON_NAMES.index(neuron_name)
        baseline = np.ones(TMAX, np.uint8) if neuron_name in TONICALLY_ACTIVE_NEURONS else np.zeros(TMAX, np.uint8)
        start = neuron["interval"][0]
        end = neuron["interval"][1]
        target_status = neuron["io"]
        # print(idx, neuron_name, baseline)
        for i in baseline:
            if target_status == "off":
                baseline[start:end] = 0
            elif target_status == "on":
                baseline[start:end] = 1

        baseline = baseline.reshape(TMAX//BIN_SIZE, BIN_SIZE)
        baseline = np.sum(baseline, axis=1,dtype=np.uint32)
        baseline = (baseline != 0).astype(np.uint8)

        condition_criteria.append((neuron_name, idx, baseline))
    condition_criteria = sorted(condition_criteria, key=lambda tup: tup[1])
    conditions.append(condition_criteria)

crit_Exp, crit_Cont = conditions

crit_indices = np.array([neu[1] for neu in crit_Cont])
crit_Exp = np.vstack([neu[2] for neu in crit_Exp])
crit_Cont = np.vstack([neu[2] for neu in crit_Cont])


In [5]:
# Takes the state at t and updates world to t+1. Returns spikes from step

@njit(parallel=False, fastmath=True, cache=True)
def step_kernel(V, U, Ibuf, t_ptr,
                a, b, vreset, d, k, vr, vt, vpeak, C, E, 
                W, alpha):
    n, L = V.size, alpha.size
    spk  = np.zeros(n, dtype=np.uint8)

    # integrate -------------------------------------------------------
    for i in range(n):
        I = Ibuf[t_ptr, i]
        dV  = (k[i]*(V[i]-vr[i])*(V[i]-vt[i]) - U[i] + I + E[i]) / C[i]
        dU  = a[i]*(b[i]*(V[i]-vr[i]) - U[i])
        V[i] += dV
        U[i] += dU
        if V[i] >= vpeak[i]:
            V[i]  = vreset[i]
            U[i] += d[i]
            spk[i] = 1          # Double check the formula to make sure it aint wonky

    # distribute PSC --------------------------------------------------
    if np.sum(spk) > 0:
        post_I = spk.astype(np.float32) @ W                   # dense GEMV
        t_next = (t_ptr + 1) % L
        for k_shift in range(L):
            Ibuf[(t_next + k_shift) % L, :] += post_I * alpha[k_shift]

    Ibuf[t_ptr,:] = 0.0
    return spk, (t_ptr + 1) % L

In [6]:
@njit(fastmath=True, cache=True)
def score_bin(curr_bin_results, crit_matrix, crit_indices, bin_idx, pass_ids):
    score = 0
    for i in range(len(crit_indices)):
        idx = crit_indices[i]
        if curr_bin_results[idx] == crit_matrix[i, bin_idx]:
            score += 1
        elif (bin_idx * BIN_SIZE > 3500) and (idx in pass_ids):
            score += 1
    return score


In [7]:
# ────────────────────────────────────────────────────────────────────
# 2.  Simulation + scoring
# ────────────────────────────────────────────────────────────────────

@njit(fastmath=True, cache=True)
def simulate(W, 
            a, b, vreset, d, k, vr, vt, vpeak, C, E, 
            alpha, cue_wave, go_wave, 
            crit_Exp, crit_Cont, crit_indices, pass_ids,
            tmax, 
            control, 
            return_full 
            ):
    
    W = np.ascontiguousarray(W)
    V = np.full(N, -60.0, np.float32)
    U = np.zeros_like(V, np.float32)
    Ibuf = np.zeros((ALPHA_L, N), dtype=np.float32)
    HIST = np.zeros((N, BIN_SIZE), np.uint8) # 99?
    if return_full:
        temp_full_hist = np.zeros((N, tmax), np.uint8) # 99?

    score = 0
    t_ptr   = 0
    bin = 0

    for t in range(tmax):

        if control == False:
            Ibuf[t_ptr,0] += cue_wave[t]
        Ibuf[t_ptr,7] += go_wave[t]
    
        spk, t_ptr = step_kernel(V, U, Ibuf, t_ptr,
                                 a, b, vreset, d, k, vr, vt, vpeak, C, E, 
                                 W, alpha)

        if return_full:
            temp_full_hist[:,t] = spk 

        cidx = t % BIN_SIZE
        HIST[:,cidx] = spk
        # bit-pack history
        if cidx == (BIN_SIZE - 1):
            curr_bin_results = (np.sum(HIST, axis=1) >= 1).astype(np.uint8)
            crits = crit_Exp if (control == False) else crit_Cont
            score += score_bin(curr_bin_results,crits, crit_indices, bin, pass_ids)
            bin += 1

    return score, (temp_full_hist if return_full else None)


In [8]:
start = perf_counter()
simulate(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
         alpha, cue_wave, go_wave,
         crit_Exp, crit_Cont, crit_indices, pass_ids,
         5000, False, False)

# print(mid-start)
# run_batch(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
#           alpha, cue_wave, go_wave,
#           crit_Exp, crit_Cont, crit_indices, pass_ids,
#           TMAX)
end = perf_counter()
print(f'Total time: {end - start:.3f}s')


Total time: 0.481s


In [ ]:
@njit(cache=True)
def run_batch(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
              alpha, cue_wave, go_wave,
              crit_Exp, crit_Cont, crit_indices, pass_ids,
              tmax, NTRIALS=100):

    total_time = 0.0
    for i in range(NTRIALS):
        s1, _ = simulate(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
                         alpha, cue_wave, go_wave,
                         crit_Exp, crit_Cont, crit_indices, pass_ids,
                         tmax, False, False)
        s2, _ = simulate(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
                         alpha, cue_wave, go_wave,
                         crit_Exp, crit_Cont, crit_indices, pass_ids,
                         tmax, True, False)
    return s1, s2

In [10]:
start = perf_counter()
s1,s2=run_batch(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
          alpha, cue_wave, go_wave,
          crit_Exp, crit_Cont, crit_indices, pass_ids,
          TMAX)
end = perf_counter()
print(f'Total time: {end - start:.3f}s')


Total time: 0.371s


# Runtimes for 10 runs
- No JIT: 6 seconds
- Step Kernel @njit(parallel=False, fastmath=True, cache=True) without prange: 0.592s
- Step Kernel @njit(parallel=False, fastmath=True, cache=True) with prange: 0.555s
- Step Kernel @njit(parallel=True, fastmath=True, cache=True) without prange: 22.615s 
- Step Kernel @njit(parallel=True, fastmath=True, cache=True) with prange: 21.592s
- Step Kernel and Score Bin JIT: 0.569s
- Step Kernel and Score Bin and Simulate JIT: HUNG, but then .388s

In [23]:
print(s1, s2)


470 498


# Config param setup
```
GA_CONFIG={
    "small": {
        "NUM_GENERATIONS" : 5,
        "POP_SIZE" : 50,
        "MUT_RATE" : 0.3,
        "MUT_SIGMA" : 0.5,
        "RANK_DEPTH" : 25,
        "ELITE_SIZE" : 5,
        "CROSSOVER_POINT" : None, # Randomly selecting all genes
        "DNA_BOUNDS" : [0,500]
    },...
}
```

In [11]:
# Generating random matrices

def initialize_population(size, upper_bound, synapses, inhibited):
    base_vector = np.ones(len(synapses), dtype=np.int32)
    for idx, conn in enumerate(synapses):
        if conn[0] in inhibited:
            base_vector[idx] *= -1
    vectors = []
    while len(vectors) < size:
        # Ensure random values are int32 to match base_vector
        random_values = np.random.randint(0, upper_bound, len(base_vector), dtype=np.int32)
        random_vector = base_vector * random_values
        # Explicitly ensure the result is int32
        random_vector = random_vector.astype(np.int32)
        vectors.append(random_vector)      
    vectors = np.array(vectors, dtype=np.int32)  # Ensure final array is int32
    return vectors

def initialize_connection_mapping(synapses: list, neuron_names:list) -> np.ndarray:
        indices = np.zeros((len(synapses), 2), dtype=np.int32)
        for idx, conn in enumerate(synapses):
            indices[idx, 0] = neuron_names.index(conn[0])  # pre-synaptic
            indices[idx, 1] = neuron_names.index(conn[1])  # post-synaptic
        return indices

@njit
def create_matrices(dna_vectors: np.array, conn_map: np.array, num_neurons: int):   
    N = num_neurons
    Ws = np.zeros((len(dna_vectors), N, N), dtype=np.float32)
    for idx, vector in enumerate(dna_vectors):
        for w, conn in zip(vector, conn_map):
            Ws[idx, conn[0], conn[1]] += w
    return Ws

gen0 = initialize_population(100,500,ACTIVE_SYNAPSES,INHIBITORY_NEURONS)
print(gen0[0])
conn_map = initialize_connection_mapping(ACTIVE_SYNAPSES, NEURON_NAMES)
print(conn_map[0])

start=perf_counter()
matrices = create_matrices(gen0, conn_map, N)
end = perf_counter()
print(end-start)

print(*matrices[0,:,:])

[ 400  167   12  246  -50  -13 -246  -32 -116 -129  -75  -51 -188 -470
 -161 -485  -99 -363 -329   51   97  310   79  105  320  346  356  357
 -126 -449 -439   55  414   17  152  485   79  112 -433 -150 -222 -343
 -298 -390  333  478 -425 -284  375  137  160  441  431]
[0 4]
0.27548570799990557
[  0. 167.   0.   0. 400.  12.   0.   0.   0.   0. 246.   0.   0.   0.] [   0.    0.  -50.    0.    0. -433.  -13.    0.    0.    0. -150. -246.
    0.    0.] [   0.    0.    0. -470.    0.    0.    0.    0.    0.    0.    0.    0.
    0. -161.] [  0.   0.   0.   0.  51.   0.   0.   0.   0.  97.   0.   0. 310.   0.] [  0. 346.   0. 152.   0. 356.   0.   0.   0. 333. 357.   0. 478. 485.] [   0. -222.  -32.    0.    0.    0. -116.    0.    0.    0. -343. -129.
    0.    0.] [   0.    0.    0. -485.    0.    0.    0.    0.    0.    0.    0.    0.
    0.  -99.] [  0.   0.   0.   0.   0.   0.   0.   0. 160.   0.   0.   0.   0.   0.] [  0.   0.   0.   0.   0.   0.   0.   0.   0. 441.   0.   0. 431.   

In [12]:

@njit(parallel=False, fastmath=True, cache=True)
def evaluate_population(population_vectors,
                            conn_map, N,
                            a, b, vreset, d, k, vr, vt, vpeak, C, E, 
                            alpha, cue_wave, go_wave, 
                            crit_Exp, crit_Cont, crit_indices, pass_ids,
                            tmax, 
                            return_full):

    population_matrices = create_matrices(population_vectors, conn_map , N)
    
    vectors_scores = np.zeros((len(population_matrices), 3), dtype=np.int32)
    for idx in range(len(population_vectors)):
        exp_score, _ = simulate(population_matrices[idx], a, b, vreset, d, k, vr, vt, vpeak, C, E,
                            alpha, cue_wave, go_wave,
                            crit_Exp, crit_Cont, crit_indices, pass_ids,
                            tmax, False, return_full)
        cont_score, _ = simulate(population_matrices[idx], a, b, vreset, d, k, vr, vt, vpeak, C, E,
                            alpha, cue_wave, go_wave,
                            crit_Exp, crit_Cont, crit_indices, pass_ids,
                            tmax, True, return_full)
        vectors_scores[idx,0] = idx  # Use index instead of full vector
        vectors_scores[idx,1] = exp_score
        vectors_scores[idx,2] = cont_score    

    return vectors_scores 

In [ ]:
tmax=5000
start=perf_counter()

pop = initialize_population(100, 500, ACTIVE_SYNAPSES,INHIBITORY_NEURONS)
conn_map = initialize_connection_mapping(ACTIVE_SYNAPSES, NEURON_NAMES)

scores = evaluate_population(pop,
                            conn_map, N,                                    
                            a, b, vreset, d, k, vr, vt, vpeak, C, E,
                            alpha, cue_wave, go_wave,
                            crit_Exp, crit_Cont, crit_indices, pass_ids,
                            tmax, False)
print("Population evaluation results:")
print("Index | Exp Score | Cont Score")
print("------|-----------|----------")
for i, (idx, exp, cont) in enumerate(scores):
    print(f"{i:5d} | {exp:9d} | {cont:10d} | {cont+exp} | {pop[i]}")



end=perf_counter()
print(end-start)


Population evaluation results:
Index | Exp Score | Cont Score
------|-----------|----------
    0 |       363 |        453 | 816 | [ 248  199  375  134 -412 -172 -368  -34 -479 -382 -305 -160 -416 -484
 -224 -155  -99 -252  -27  162  185  309  447  429  117  246  133  323
 -494 -384 -367  264   19  105   46  352    0  359 -488 -229  -51  -65
 -346 -291  445  211 -387 -281   10   32  371  236   98]
    1 |       316 |        363 | 679 | [ 225  108  454  208  -32 -418 -212  -27  -46  -99  -33 -247 -211 -431
 -488 -192 -172 -212  -67  480  453  343  426  118  383  167  169  474
 -333  -49  -23  419   24  365  459  284  483  243  -79 -419 -163 -490
 -406  -15  490  378 -387 -175  122  332  317  348   33]
    2 |       332 |        370 | 702 | [ 142  303  445  330 -460 -310  -64  -73 -283 -127 -427  -62 -201 -160
  -21 -344 -178 -138 -232  489  164  436  481  484   70    0  256  408
 -366 -424 -437  173  397   65  152  153  288  373 -231 -198  -82 -393
 -498 -430  308  373 -401  -52  234  1

In [14]:
dpops = [{"dna": pop[i], "dna_score": exp+cont} for i, (idx, exp, cont) in enumerate(scores)]

print(f"Created {len(dpops)} population records")
print(f"Sample record: {dpops[0]}")
print(f"Type of dpops: {type(dpops)}")

Created 100 population records
Sample record: {'dna': array([ 248,  199,  375,  134, -412, -172, -368,  -34, -479, -382, -305,
       -160, -416, -484, -224, -155,  -99, -252,  -27,  162,  185,  309,
        447,  429,  117,  246,  133,  323, -494, -384, -367,  264,   19,
        105,   46,  352,    0,  359, -488, -229,  -51,  -65, -346, -291,
        445,  211, -387, -281,   10,   32,  371,  236,   98], dtype=int32), 'dna_score': np.int32(816)}
Type of dpops: <class 'list'>


# Now to code all the genetic mutation stuff


In [15]:

import random
from typing import List, Tuple

_ORIGIN_IDX = np.array([NEURON_NAMES.index(o) for o, _ in ACTIVE_SYNAPSES], dtype=np.int16)
_TARGET_IDX = np.array([NEURON_NAMES.index(t) for _, t in ACTIVE_SYNAPSES], dtype=np.int16)

_INHIB_MASK  = np.isin(_ORIGIN_IDX, [NEURON_NAMES.index(n) for n in INHIBITORY_NEURONS])

# ------------------------------------------------------------------
# 2.  Operators
# ------------------------------------------------------------------

def uniform_crossover(p1: np.ndarray, p2: np.ndarray, swap_p: float = 0.5) -> np.ndarray:
    """Per‑gene uniform crossover."""
    mask = np.random.rand(p1.size) < swap_p
    child = np.where(mask, p1, p2)
    return child.astype(np.int32)


def mutate_gauss(dna: np.ndarray, sigma: float, bounds: Tuple[int, int]) -> np.ndarray:
    """Gaussian mutation with automatic rounding and sign fix."""
    dna = dna.astype(np.float32) + np.dot(np.random.normal(0, sigma, size=dna.size), dna.astype(np.float32))
    low, high = bounds
    dna = np.clip(np.round(dna), -high, high)
    dna[_INHIB_MASK] = -np.abs(dna[_INHIB_MASK])
    dna[~_INHIB_MASK] =  np.abs(dna[~_INHIB_MASK])
    return dna.astype(np.int32)

# ------------------------------------------------------------------
# 3.  Population spawning
# ------------------------------------------------------------------

def _tournament(pop: List[dict], k: int) -> np.ndarray:
    """Return winner DNA from k‑sized tournament (higher score wins)."""
    contenders = random.sample(pop, k)
    return max(contenders, key=lambda r: r["dna_score"]) ["dna"]


def _hamming(a: np.ndarray, b: np.ndarray) -> int:
    return int(np.sum(a != b))


def spawn_next_population(pop_records: List[dict], cfg: dict) -> List[np.ndarray]:
    pop_size   = cfg["POP_SIZE"]
    bounds     = tuple(cfg["DNA_BOUNDS"])
    elite_n    = cfg["ELITE_SIZE"]
    rank_depth = cfg["RANK_DEPTH"]
    sigma      = cfg["MUT_SIGMA"]
    mut_rate   = cfg["MUT_RATE"]

    pop_records.sort(key=lambda r: r["dna_score"], reverse=True)
    elites = [r["dna"] for r in pop_records[:elite_n]]
    next_pop = elites.copy()

    # keep‑distance niching threshold (5 % of chromosome length)
    niche_thresh = 0.01 * _ORIGIN_IDX.size

    while len(next_pop) < pop_size:
        p1 = _tournament(pop_records[:rank_depth], 3)
        p2 = _tournament(pop_records[:rank_depth], 3)
        child = uniform_crossover(p1, p2, swap_p=0.5)
        if np.random.rand() < mut_rate:
            child = mutate_gauss(child, sigma, bounds)

        if all(_hamming(child, dna) > niche_thresh for dna in next_pop):
            next_pop.append(child)

    return next_pop


In [88]:
# Test the spawn_next_population function
cfg = "small"
el = 0
print(f"Testing spawn_next_population with config: {cfg}")
print(f"Population size before: {len(dpops)}")
# print(f"Top 5 scores: {[record['dna'] for record in sorted(dpops, key=lambda r: r['dna_score'], reverse=True)[:10]]}")
newrec= [record['dna'] for record in sorted(dpops, key=lambda r: r['dna_score'], reverse=True)[:10]]
print(newrec[el])
# This should now work without the TypeError
pop1 = spawn_next_population(dpops, GA_CONFIG[cfg])
print(f"Next generation population size: {len(pop1)}")
print(f"Sample DNA from next generation: {pop1[el][:10]}...")  # Show first 10 genes


Testing spawn_next_population with config: small
Population size before: 100
[   4  334  321  297    0 -485   -9  -10 -480 -448  -98  -25 -148 -155
 -461  -67 -466 -110  -82  356   46  443  454  281   20  137  474  262
 -304 -129 -161  192   54  250  229   26  380  372  -98 -176 -225 -271
 -461 -311  224  336 -281 -456   31  437   21  488  375]
Next generation population size: 100
Sample DNA from next generation: [   4  334  321  297    0 -485   -9  -10 -480 -448]...


In [17]:
t00 = perf_counter()
cfg="medium"
tmax=5000
pop = initialize_population(GA_CONFIG[cfg]["POP_SIZE"], 
                        GA_CONFIG[cfg]["DNA_BOUNDS"][1], 
                        ACTIVE_SYNAPSES,
                        INHIBITORY_NEURONS
                        )
t01 = perf_counter()
print(f'{t01-t00=}')
conn_map = initialize_connection_mapping(ACTIVE_SYNAPSES, NEURON_NAMES)
t02 = perf_counter()
print(f'{t02-t01=}')
for gen in range(GA_CONFIG[cfg]["NUM_GENERATIONS"]):
    t1 = perf_counter()
    scores = evaluate_population(pop,
                                conn_map, N,                                    
                                a, b, vreset, d, k, vr, vt, vpeak, C, E,
                                alpha, cue_wave, go_wave,
                                crit_Exp, crit_Cont, crit_indices, pass_ids,
                                tmax, False)
    t2 = perf_counter()
    print(f'{t2-t1=}')
    for i, (idx, exp, cont) in enumerate(scores):
        print(f" {gen=} | {i:5d} | {exp:9d} | {cont:10d} | {cont+exp}")

    dpops = [{"dna": pop[i], "dna_score": exp+cont} for i, (idx, exp, cont) in enumerate(scores)]
    t3 = perf_counter()
    print(f'{t3-t2=}')
    pop = spawn_next_population(dpops, GA_CONFIG[cfg])
    t4 = perf_counter()
    print(f'{t4-t3=}')

t01-t00=0.0023202080046758056
t02-t01=0.0011748330143745989
t2-t1=24.193022207997274
 gen=0 |     0 |       315 |        375 | 690
 gen=0 |     1 |       326 |        352 | 678
 gen=0 |     2 |       375 |        500 | 875
 gen=0 |     3 |       327 |        359 | 686
 gen=0 |     4 |       332 |        400 | 732
 gen=0 |     5 |       332 |        366 | 698
 gen=0 |     6 |       314 |        369 | 683
 gen=0 |     7 |       337 |        415 | 752
 gen=0 |     8 |       336 |        406 | 742
 gen=0 |     9 |       349 |        429 | 778
 gen=0 |    10 |       249 |        199 | 448
 gen=0 |    11 |       351 |        500 | 851
 gen=0 |    12 |       327 |        414 | 741
 gen=0 |    13 |       336 |        416 | 752
 gen=0 |    14 |       317 |        382 | 699
 gen=0 |    15 |       329 |        497 | 826
 gen=0 |    16 |       248 |        161 | 409
 gen=0 |    17 |       362 |        452 | 814
 gen=0 |    18 |       319 |        402 | 721
 gen=0 |    19 |       301 |        317 |

In [18]:
totals = [scores[i][1]+scores[i][2] for i in range(len(scores))]
print(totals)

[np.int32(920), np.int32(920), np.int32(920), np.int32(920), np.int32(920), np.int32(920), np.int32(638), np.int32(920), np.int32(920), np.int32(594), np.int32(920), np.int32(920), np.int32(920), np.int32(594), np.int32(696), np.int32(920), np.int32(744), np.int32(920), np.int32(920), np.int32(920), np.int32(920), np.int32(920), np.int32(778), np.int32(594), np.int32(920), np.int32(920), np.int32(594), np.int32(920), np.int32(698), np.int32(920), np.int32(920), np.int32(526), np.int32(920), np.int32(920), np.int32(920), np.int32(920), np.int32(740), np.int32(920), np.int32(590), np.int32(920), np.int32(920), np.int32(920), np.int32(920), np.int32(614), np.int32(776), np.int32(920), np.int32(920), np.int32(526), np.int32(920), np.int32(920), np.int32(920), np.int32(908), np.int32(920), np.int32(920), np.int32(526), np.int32(920), np.int32(920), np.int32(920), np.int32(920), np.int32(920), np.int32(920), np.int32(920), np.int32(694), np.int32(596), np.int32(920), np.int32(920), np.int32(

In [31]:
# Enhanced Multiprocessing GA Implementation with Complete Data Saving
import multiprocessing as mp
from multiprocessing import Pool
import os
from time import perf_counter
from datetime import datetime
import pickle
import json
import sqlite3
import pandas as pd

def run_single_ga_instance(args):
    """
    Run a single genetic algorithm instance.
    This function will be executed by each process.
    
    Args:
        args: tuple containing (process_id, config_name, num_generations, random_seed, tmax, save_all_data)
    
    Returns:
        dict: Results from the GA run including ALL DNA vectors and scores
    """
    process_id, config_name, num_generations, random_seed, tmax, save_all_data = args
    
    # Set random seed for this process to ensure different initial populations
    np.random.seed(random_seed)
    
    # Print process start info
    print(f"Process {process_id}: Starting GA with seed {random_seed}")
    
    # Initialize population for this process
    cfg = GA_CONFIG[config_name]
    pop = initialize_population(cfg["POP_SIZE"], 
                              cfg["DNA_BOUNDS"][1], 
                              ACTIVE_SYNAPSES,
                              INHIBITORY_NEURONS)
    
    conn_map = initialize_connection_mapping(ACTIVE_SYNAPSES, NEURON_NAMES)
    
    # Track the best individual across generations
    best_individual = None
    best_score = -1
    generation_scores = []
    
    # NEW: Store ALL DNA vectors and scores if requested
    all_individuals = [] if save_all_data else None
    
    start_time = perf_counter()
    
    # Run the genetic algorithm
    for gen in range(num_generations):
        # Evaluate population
        scores = evaluate_population(pop,
                                   conn_map, N,                                    
                                   a, b, vreset, d, k, vr, vt, vpeak, C, E,
                                   alpha, cue_wave, go_wave,
                                   crit_Exp, crit_Cont, crit_indices, pass_ids,
                                   tmax, False)
        
        # Create population records
        dpops = [{"dna": pop[i], "dna_score": exp+cont} for i, (idx, exp, cont) in enumerate(scores)]
        
        # NEW: Save all individuals from this generation
        if save_all_data:
            for i, (idx, exp_score, cont_score) in enumerate(scores):
                individual_record = {
                    'process_id': process_id,
                    'generation': gen,
                    'individual_id': i,
                    'dna': pop[i].copy(),  # Copy to avoid reference issues
                    'exp_score': int(exp_score),
                    'cont_score': int(cont_score),
                    'total_score': int(exp_score + cont_score),
                    'random_seed': random_seed,
                    'config_name': config_name,
                    'timestamp': datetime.now().isoformat()
                }
                all_individuals.append(individual_record)
        
        # Track best individual
        current_best = max(dpops, key=lambda x: x["dna_score"])
        if current_best["dna_score"] > best_score:
            best_score = current_best["dna_score"]
            best_individual = current_best["dna"].copy()
        
        # Store generation statistics
        scores_list = [record["dna_score"] for record in dpops]
        generation_scores.append({
            'generation': gen,
            'best': max(scores_list),
            'mean': np.mean(scores_list),
            'std': np.std(scores_list)
        })
        
        # Print progress for this process
        if gen % 10 == 0 or gen == num_generations - 1:
            print(f"Process {process_id}: Gen {gen}, Best: {max(scores_list)}, Mean: {np.mean(scores_list):.1f}")
        
        # Generate next population (except for last generation)
        if gen < num_generations - 1:
            pop = spawn_next_population(dpops, cfg)
    
    end_time = perf_counter()
    runtime = end_time - start_time
    
    print(f"Process {process_id}: Completed in {runtime:.2f}s, Final best score: {best_score}")
    
    result = {
        'process_id': process_id,
        'random_seed': random_seed,
        'best_individual': best_individual,
        'best_score': best_score,
        'generation_scores': generation_scores,
        'runtime': runtime,
        'final_population_size': len(pop),
        'config_name': config_name
    }
    
    # Add all individuals if requested
    if save_all_data:
        result['all_individuals'] = all_individuals
        print(f"Process {process_id}: Saved {len(all_individuals)} individuals")
    
    return result


In [32]:
def run_parallel_ga_with_data_saving(config_name="F", num_processes=None, num_generations=None, 
                                     tmax=5000, save_all_data=True, output_dir="results"):
    """
    Run multiple GA instances in parallel and save all data for analysis.
    
    Args:
        config_name: Configuration to use from GA_CONFIG
        num_processes: Number of parallel processes (default: CPU count)
        num_generations: Override number of generations (default: from config)
        tmax: Maximum time steps for simulation
        save_all_data: Whether to save all DNA vectors and scores
        output_dir: Directory to save results
    
    Returns:
        tuple: (results, total_runtime, file_paths)
    """
    if num_processes is None:
        num_processes = mp.cpu_count()
    
    cfg = GA_CONFIG[config_name]
    if num_generations is None:
        num_generations = cfg["NUM_GENERATIONS"]
    
    # Create output directory
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = f"{output_dir}/parallel_ga_{config_name}_{timestamp}"
    os.makedirs(run_dir, exist_ok=True)
    
    print(f"Starting {num_processes} parallel GA processes")
    print(f"Config: {config_name}, Generations: {num_generations}, Population size: {cfg['POP_SIZE']}")
    print(f"Total individuals to evaluate: {num_processes * cfg['POP_SIZE'] * num_generations}")
    print(f"Results will be saved to: {run_dir}")
    print("-" * 60)
    
    # Create arguments for each process with different random seeds
    base_seed = int(datetime.now().timestamp())
    process_args = []
    for i in range(num_processes):
        seed = base_seed + i * 1000  # Ensure seeds are well separated
        process_args.append((i, config_name, num_generations, seed, tmax, save_all_data))
    
    start_time = perf_counter()
    
    # Run processes in parallel
    with Pool(processes=num_processes) as pool:
        results = pool.map(run_single_ga_instance, process_args)
    
    end_time = perf_counter()
    total_runtime = end_time - start_time
    
    print("\n" + "="*60)
    print(f"All processes completed in {total_runtime:.2f}s")
    
    # Find overall best result
    best_result = max(results, key=lambda x: x['best_score'])
    print(f"Overall best score: {best_result['best_score']} (Process {best_result['process_id']})")
    
    # Save results
    file_paths = save_results_to_files(results, run_dir, config_name, save_all_data)
    
    return results, total_runtime, file_paths

def save_results_to_files(results, run_dir, config_name, save_all_data):
    """
    Save results to multiple file formats for analysis.
    
    Returns:
        dict: Paths to saved files
    """
    file_paths = {}
    
    # 1. Save complete results as pickle (most comprehensive)
    pickle_path = os.path.join(run_dir, "complete_results.pkl")
    with open(pickle_path, 'wb') as f:
        pickle.dump(results, f)
    file_paths['pickle'] = pickle_path
    print(f"✅ Saved complete results to: {pickle_path}")
    
    # 2. Save summary as JSON (human-readable)
    summary_data = {
        'config_name': config_name,
        'num_processes': len(results),
        'timestamp': datetime.now().isoformat(),
        'total_individuals': sum(len(r.get('all_individuals', [])) for r in results),
        'best_scores': [r['best_score'] for r in results],
        'process_summary': [
            {
                'process_id': r['process_id'],
                'best_score': r['best_score'],
                'runtime': r['runtime'],
                'random_seed': r['random_seed']
            } for r in results
        ]
    }
    
    json_path = os.path.join(run_dir, "summary.json")
    with open(json_path, 'w') as f:
        json.dump(summary_data, f, indent=2)
    file_paths['json'] = json_path
    print(f"✅ Saved summary to: {json_path}")
    
    # 3. Save all individuals to CSV and database (if data was collected)
    if save_all_data and any('all_individuals' in r for r in results):
        # Collect all individuals across all processes
        all_individuals = []
        for result in results:
            if 'all_individuals' in result:
                all_individuals.extend(result['all_individuals'])
        
        if all_individuals:
            # Convert to DataFrame for analysis
            df_data = []
            for ind in all_individuals:
                row = {
                    'process_id': ind['process_id'],
                    'generation': ind['generation'],
                    'individual_id': ind['individual_id'],
                    'exp_score': ind['exp_score'],
                    'cont_score': ind['cont_score'],
                    'total_score': ind['total_score'],
                    'random_seed': ind['random_seed'],
                    'config_name': ind['config_name'],
                    'timestamp': ind['timestamp']
                }
                # Add DNA genes as separate columns
                for i, gene in enumerate(ind['dna']):
                    row[f'gene_{i:02d}'] = int(gene)
                df_data.append(row)
            
            df = pd.DataFrame(df_data)
            
            # Save as CSV
            csv_path = os.path.join(run_dir, "all_individuals.csv")
            df.to_csv(csv_path, index=False)
            file_paths['csv'] = csv_path
            print(f"✅ Saved {len(df)} individuals to CSV: {csv_path}")
            
            # Save to SQLite database
            db_path = os.path.join(run_dir, "ga_results.db")
            conn = sqlite3.connect(db_path)
            df.to_sql('individuals', conn, if_exists='replace', index=False)
            
            # Create summary table
            summary_df = pd.DataFrame([
                {
                    'process_id': r['process_id'],
                    'best_score': r['best_score'],
                    'runtime': r['runtime'],
                    'random_seed': r['random_seed'],
                    'config_name': r['config_name']
                } for r in results
            ])
            summary_df.to_sql('process_summary', conn, if_exists='replace', index=False)
            conn.close()
            file_paths['database'] = db_path
            print(f"✅ Saved to SQLite database: {db_path}")
            
            # Save best individuals separately
            best_individuals = df.loc[df.groupby('process_id')['total_score'].idxmax()]
            best_csv_path = os.path.join(run_dir, "best_individuals.csv")
            best_individuals.to_csv(best_csv_path, index=False)
            file_paths['best_csv'] = best_csv_path
            print(f"✅ Saved {len(best_individuals)} best individuals to: {best_csv_path}")
    
    return file_paths


In [33]:
def load_and_analyze_results(results_dir):
    """
    Load and analyze saved GA results for comprehensive analysis.
    
    Args:
        results_dir: Directory containing the saved results
    
    Returns:
        dict: Analysis results including DataFrames and statistics
    """
    analysis = {}
    
    # Load pickle file (most complete data)
    pickle_path = os.path.join(results_dir, "complete_results.pkl")
    if os.path.exists(pickle_path):
        with open(pickle_path, 'rb') as f:
            results = pickle.load(f)
        analysis['raw_results'] = results
        print(f"📊 Loaded complete results: {len(results)} processes")
    
    # Load CSV data if available
    csv_path = os.path.join(results_dir, "all_individuals.csv")
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        analysis['individuals_df'] = df
        print(f"📊 Loaded individuals data: {len(df)} individuals")
        
        # Basic statistics
        analysis['stats'] = {
            'total_individuals': len(df),
            'unique_processes': df['process_id'].nunique(),
            'generations': df['generation'].nunique(),
            'best_score': df['total_score'].max(),
            'mean_score': df['total_score'].mean(),
            'std_score': df['total_score'].std(),
            'score_range': df['total_score'].max() - df['total_score'].min()
        }
        
        # Per-process analysis
        process_stats = df.groupby('process_id').agg({
            'total_score': ['max', 'mean', 'std'],
            'generation': 'max'
        }).round(2)
        analysis['process_stats'] = process_stats
        
        # Evolution over generations
        gen_stats = df.groupby(['process_id', 'generation']).agg({
            'total_score': ['max', 'mean', 'std']
        }).round(2)
        analysis['generation_stats'] = gen_stats
        
        # Best individuals per generation
        best_per_gen = df.loc[df.groupby(['process_id', 'generation'])['total_score'].idxmax()]
        analysis['best_per_generation'] = best_per_gen
        
        print(f"📈 Analysis complete:")
        print(f"   Total individuals: {analysis['stats']['total_individuals']}")
        print(f"   Best score found: {analysis['stats']['best_score']}")
        print(f"   Mean score: {analysis['stats']['mean_score']:.2f}")
        print(f"   Score range: {analysis['stats']['score_range']}")
    
    # Load database if available
    db_path = os.path.join(results_dir, "ga_results.db")
    if os.path.exists(db_path):
        conn = sqlite3.connect(db_path)
        analysis['database_connection'] = db_path
        
        # Get table info
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = cursor.fetchall()
        print(f"🗄️  Database available with tables: {[t[0] for t in tables]}")
        conn.close()
    
    return analysis

def visualize_ga_results(analysis):
    """
    Create visualizations of the GA results.
    """
    import matplotlib.pyplot as plt
    
    if 'individuals_df' not in analysis:
        print("❌ No individual data available for visualization")
        return
    
    df = analysis['individuals_df']
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Genetic Algorithm Results Analysis', fontsize=16)
    
    # 1. Score distribution
    axes[0, 0].hist(df['total_score'], bins=50, alpha=0.7, edgecolor='black')
    axes[0, 0].set_xlabel('Total Score')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Score Distribution')
    axes[0, 0].axvline(df['total_score'].mean(), color='red', linestyle='--', label=f'Mean: {df["total_score"].mean():.1f}')
    axes[0, 0].legend()
    
    # 2. Evolution over generations (best per generation)
    for process_id in df['process_id'].unique():
        process_data = df[df['process_id'] == process_id]
        gen_best = process_data.groupby('generation')['total_score'].max()
        axes[0, 1].plot(gen_best.index, gen_best.values, alpha=0.7, label=f'Process {process_id}')
    
    axes[0, 1].set_xlabel('Generation')
    axes[0, 1].set_ylabel('Best Score')
    axes[0, 1].set_title('Evolution of Best Scores')
    axes[0, 1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # 3. Process comparison (boxplot)
    process_scores = [df[df['process_id'] == pid]['total_score'].values 
                     for pid in sorted(df['process_id'].unique())]
    axes[1, 0].boxplot(process_scores, labels=[f'P{i}' for i in sorted(df['process_id'].unique())])
    axes[1, 0].set_xlabel('Process ID')
    axes[1, 0].set_ylabel('Total Score')
    axes[1, 0].set_title('Score Distribution by Process')
    
    # 4. Score correlation between exp and cont
    axes[1, 1].scatter(df['exp_score'], df['cont_score'], alpha=0.5)
    axes[1, 1].set_xlabel('Experimental Score')
    axes[1, 1].set_ylabel('Control Score')
    axes[1, 1].set_title('Experimental vs Control Scores')
    
    # Add correlation coefficient
    correlation = df['exp_score'].corr(df['cont_score'])
    axes[1, 1].text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
                   transform=axes[1, 1].transAxes, bbox=dict(boxstyle="round", facecolor='wheat'))
    
    plt.tight_layout()
    plt.show()
    
    return fig

def query_database(db_path, query):
    """
    Execute SQL queries on the results database.
    
    Args:
        db_path: Path to the SQLite database
        query: SQL query string
    
    Returns:
        pandas.DataFrame: Query results
    """
    conn = sqlite3.connect(db_path)
    result = pd.read_sql_query(query, conn)
    conn.close()
    return result

# Example queries for analysis
EXAMPLE_QUERIES = {
    "top_10_individuals": """
        SELECT process_id, generation, individual_id, total_score, exp_score, cont_score
        FROM individuals 
        ORDER BY total_score DESC 
        LIMIT 10
    """,
    
    "best_per_process": """
        SELECT process_id, MAX(total_score) as best_score, 
               AVG(total_score) as avg_score,
               COUNT(*) as total_individuals
        FROM individuals 
        GROUP BY process_id 
        ORDER BY best_score DESC
    """,
    
    "evolution_summary": """
        SELECT generation, 
               MAX(total_score) as gen_best,
               AVG(total_score) as gen_avg,
               COUNT(*) as individuals_count
        FROM individuals 
        GROUP BY generation 
        ORDER BY generation
    """,
    
    "high_performers": """
        SELECT * FROM individuals 
        WHERE total_score > (SELECT AVG(total_score) + STD(total_score) FROM individuals)
        ORDER BY total_score DESC
    """
}


In [34]:
# EXAMPLE USAGE: Run multiprocessing GA with complete data saving

# Example 1: Quick test run with data saving
print("=" * 70)
print("EXAMPLE 1: Quick Test Run with Complete Data Saving")
print("=" * 70)

results, runtime, file_paths = run_parallel_ga_with_data_saving(
    config_name="small",      # Small configuration for testing
    num_processes=4,          # Use 4 processes
    num_generations=5,        # Just 5 generations for demo
    tmax=5000,               # Shorter simulation time
    save_all_data=True,      # Save ALL DNA vectors and scores
    output_dir="results"     # Save to results directory
)

print(f"\n🎉 Test run completed!")
print(f"📁 Files saved:")
for format_type, path in file_paths.items():
    print(f"   {format_type}: {path}")

# Show some basic statistics
if 'csv' in file_paths:
    df = pd.read_csv(file_paths['csv'])
    print(f"\n📊 Quick stats:")
    print(f"   Total individuals evaluated: {len(df)}")
    print(f"   Best score found: {df['total_score'].max()}")
    print(f"   Mean score: {df['total_score'].mean():.2f}")
    print(f"   Processes used: {df['process_id'].nunique()}")
    print(f"   Generations: {df['generation'].nunique()}")


EXAMPLE 1: Quick Test Run with Complete Data Saving
Starting 4 parallel GA processes
Config: small, Generations: 5, Population size: 100
Total individuals to evaluate: 2000
Results will be saved to: results/parallel_ga_small_20250726_002416
------------------------------------------------------------


Process SpawnPoolWorker-2:
Process SpawnPoolWorker-3:
Process SpawnPoolWorker-1:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/local/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/usr/local/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/queues.py", line 389, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'run_

KeyboardInterrupt: 

In [40]:
# ANALYSIS EXAMPLES: How to analyze your saved results

print("=" * 70)
print("ANALYSIS EXAMPLES")
print("=" * 70)

# Find the most recent results directory
import glob
result_dirs = glob.glob("results/parallel_ga_*")
if result_dirs:
    latest_dir = max(result_dirs, key=os.path.getctime)
    print(f"📂 Analyzing results from: {latest_dir}")
    
    # Load and analyze the results
    analysis = load_and_analyze_results(latest_dir)
    
    # Show detailed statistics
    if 'stats' in analysis:
        print(f"\n📈 Detailed Statistics:")
        for key, value in analysis['stats'].items():
            if isinstance(value, float):
                print(f"   {key}: {value:.2f}")
            else:
                print(f"   {key}: {value}")
    
    # Show best individuals from each process
    if 'individuals_df' in analysis:
        df = analysis['individuals_df']
        print(f"\n🏆 Best individual per process:")
        best_per_process = df.loc[df.groupby('process_id')['total_score'].idxmax()]
        for _, row in best_per_process.iterrows():
            print(f"   Process {row['process_id']}: Score {row['total_score']} (Gen {row['generation']})")
        
        # Show evolution of the best process
        best_process = best_per_process.loc[best_per_process['total_score'].idxmax()]
        print(f"\n🚀 Evolution of best process (Process {best_process['process_id']}):")
        process_data = df[df['process_id'] == best_process['process_id']]
        gen_evolution = process_data.groupby('generation')['total_score'].agg(['max', 'mean']).round(2)
        print(gen_evolution)
    
    # Database query examples
    if 'database_connection' in analysis:
        db_path = analysis['database_connection']
        print(f"\n🗄️  Database Query Examples:")
        
        # Top 5 individuals
        top_5 = query_database(db_path, EXAMPLE_QUERIES['top_10_individuals'].replace('10', '5'))
        print(f"\n   Top 5 individuals:")
        print(top_5)
        
        # Evolution summary
        evolution = query_database(db_path, EXAMPLE_QUERIES['evolution_summary'])
        print(f"\n   Evolution by generation:")
        print(evolution)

else:
    print("❌ No results found. Run the GA first!")

print(f"\n💡 TIP: You can run visualizations with:")
print(f"   fig = visualize_ga_results(analysis)")
print(f"\n💡 TIP: Query the database with custom SQL:")
print(f"   result = query_database(db_path, 'YOUR SQL QUERY')")


ANALYSIS EXAMPLES
📂 Analyzing results from: results/parallel_ga_small_20250726_002416

💡 TIP: You can run visualizations with:
   fig = visualize_ga_results(analysis)

💡 TIP: Query the database with custom SQL:
   result = query_database(db_path, 'YOUR SQL QUERY')


# Letting Cursor take over....

In [41]:
# FIXED VERSION: Multiprocessing that works in Jupyter notebooks
import multiprocessing as mp
from multiprocessing import Pool
import os
from time import perf_counter
from datetime import datetime
import pickle
import json
import sqlite3
import pandas as pd

# Solution 1: Use 'fork' method on macOS (works better with Jupyter)
def setup_multiprocessing():
    """Setup multiprocessing to work in Jupyter notebooks"""
    try:
        # Try to set fork method (works on macOS/Linux)
        mp.set_start_method('fork')
        print("✅ Using 'fork' multiprocessing method")
    except RuntimeError:
        # If fork is not available or already set
        print(f"ℹ️  Using multiprocessing method: {mp.get_start_method()}")

def run_single_ga_worker(args):
    """
    Worker function for multiprocessing - designed to work in Jupyter notebooks.
    This function contains all the logic and doesn't rely on external function calls.
    """
    process_id, config_name, num_generations, random_seed, tmax, save_all_data = args
    
    # Import required modules inside worker (ensures they're available)
    import numpy as np
    from time import perf_counter
    from datetime import datetime
    
    # Set random seed for this process
    np.random.seed(random_seed)
    print(f"Process {process_id}: Starting GA with seed {random_seed}")
    
    # Get configuration
    cfg = GA_CONFIG[config_name]
    
    # Initialize population for this process
    pop = initialize_population(cfg["POP_SIZE"], 
                              cfg["DNA_BOUNDS"][1], 
                              ACTIVE_SYNAPSES,
                              INHIBITORY_NEURONS)
    
    conn_map = initialize_connection_mapping(ACTIVE_SYNAPSES, NEURON_NAMES)
    
    # Initialize necessary simulation parameters (copied from your notebook)
    N_local = len(NEURON_NAMES)
    ALPHA_L = 250
    td = np.arange(1, ALPHA_L + 1, dtype=np.float32)
    alpha_local = (td / 30) * np.exp((30 - td) / 30)   

    cue_wave_local = np.zeros(tmax, dtype=np.float32)
    go_wave_local = np.zeros_like(cue_wave_local)
    cue_wave_local[EPOCHS['sample'][0]:EPOCHS['sample'][1]] = CUE_STRENGTH
    go_wave_local[EPOCHS['response'][0]:EPOCHS['response'][0] + GO_DURATION] = GO_STRENGTH

    # Create criterion matrices (copied from your setup)
    conditions = []
    for condition in CRITERIA:
        condition_criteria = []
        for neuron_name, neuron in CRITERIA[condition].items():
            idx = NEURON_NAMES.index(neuron_name)
            baseline = np.ones(tmax, np.uint8) if neuron_name in TONICALLY_ACTIVE_NEURONS else np.zeros(tmax, np.uint8)
            start = neuron["interval"][0]
            end = neuron["interval"][1]
            target_status = neuron["io"]
            
            if target_status == "off":
                baseline[start:end] = 0
            elif target_status == "on":
                baseline[start:end] = 1

            baseline = baseline.reshape(tmax//BIN_SIZE, BIN_SIZE)
            baseline = np.sum(baseline, axis=1, dtype=np.uint32)
            baseline = (baseline != 0).astype(np.uint8)

            condition_criteria.append((neuron_name, idx, baseline))
        condition_criteria = sorted(condition_criteria, key=lambda tup: tup[1])
        conditions.append(condition_criteria)

    crit_Exp_local, crit_Cont_local = conditions
    crit_indices_local = np.array([neu[1] for neu in crit_Cont_local])
    crit_Exp_local = np.vstack([neu[2] for neu in crit_Exp_local])
    crit_Cont_local = np.vstack([neu[2] for neu in crit_Cont_local])
    
    pass_ids_local = [NEURON_NAMES.index(x) for x in ["VMresp", "ALMresp", "SNR3"]]
    pass_ids_local = np.array(pass_ids_local)
    
    # Track results
    best_individual = None
    best_score = -1
    generation_scores = []
    all_individuals = [] if save_all_data else None
    
    start_time = perf_counter()
    
    # Main GA loop
    for gen in range(num_generations):
        # Evaluate population using your existing function
        scores = evaluate_population(pop,
                                   conn_map, N_local,                                    
                                   a, b, vreset, d, k, vr, vt, vpeak, C, E,
                                   alpha_local, cue_wave_local, go_wave_local,
                                   crit_Exp_local, crit_Cont_local, crit_indices_local, pass_ids_local,
                                   tmax, False)
        
        # Create population records
        dpops = [{"dna": pop[i], "dna_score": exp+cont} for i, (idx, exp, cont) in enumerate(scores)]
        
        # Save all individuals if requested
        if save_all_data:
            for i, (idx, exp_score, cont_score) in enumerate(scores):
                individual_record = {
                    'process_id': process_id,
                    'generation': gen,
                    'individual_id': i,
                    'dna': pop[i].copy(),
                    'exp_score': int(exp_score),
                    'cont_score': int(cont_score),
                    'total_score': int(exp_score + cont_score),
                    'random_seed': random_seed,
                    'config_name': config_name,
                    'timestamp': datetime.now().isoformat()
                }
                all_individuals.append(individual_record)
        
        # Track best individual
        current_best = max(dpops, key=lambda x: x["dna_score"])
        if current_best["dna_score"] > best_score:
            best_score = current_best["dna_score"]
            best_individual = current_best["dna"].copy()
        
        # Store generation statistics
        scores_list = [record["dna_score"] for record in dpops]
        generation_scores.append({
            'generation': gen,
            'best': max(scores_list),
            'mean': np.mean(scores_list),
            'std': np.std(scores_list)
        })
        
        # Print progress
        if gen % 5 == 0 or gen == num_generations - 1:
            print(f"Process {process_id}: Gen {gen:3d}, Best: {max(scores_list):4d}, Mean: {np.mean(scores_list):6.1f}")
        
        # Generate next population (except for last generation)
        if gen < num_generations - 1:
            pop = spawn_next_population(dpops, cfg)
    
    end_time = perf_counter()
    runtime = end_time - start_time
    
    print(f"Process {process_id}: Completed in {runtime:.2f}s, Final best: {best_score}")
    
    result = {
        'process_id': process_id,
        'random_seed': random_seed,
        'best_individual': best_individual,
        'best_score': best_score,
        'generation_scores': generation_scores,
        'runtime': runtime,
        'final_population_size': len(pop),
        'config_name': config_name
    }
    
    if save_all_data:
        result['all_individuals'] = all_individuals
        print(f"Process {process_id}: Saved {len(all_individuals)} individuals")
    
    return result


In [42]:
def run_parallel_ga_fixed(config_name="F", num_processes=None, num_generations=None, 
                          tmax=5000, save_all_data=True, output_dir="results"):
    """
    FIXED VERSION: Run multiple GA instances in parallel - works in Jupyter notebooks.
    """
    # Setup multiprocessing method
    setup_multiprocessing()
    
    if num_processes is None:
        num_processes = mp.cpu_count()
    
    cfg = GA_CONFIG[config_name]
    if num_generations is None:
        num_generations = cfg["NUM_GENERATIONS"]
    
    print(f"Starting {num_processes} parallel GA processes")
    print(f"Config: {config_name}, Generations: {num_generations}, Population size: {cfg['POP_SIZE']}")
    print(f"Total individuals to evaluate: {num_processes * cfg['POP_SIZE'] * num_generations}")
    
    if save_all_data:
        # Create output directory
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        run_dir = f"{output_dir}/parallel_ga_{config_name}_{timestamp}"
        os.makedirs(run_dir, exist_ok=True)
        print(f"Results will be saved to: {run_dir}")
    
    print("-" * 60)
    
    # Create arguments for each process with different random seeds
    base_seed = int(datetime.now().timestamp())
    process_args = []
    for i in range(num_processes):
        seed = base_seed + i * 1000  # Ensure seeds are well separated
        process_args.append((i, config_name, num_generations, seed, tmax, save_all_data))
    
    start_time = perf_counter()
    
    # Run processes in parallel using the fixed worker function
    with Pool(processes=num_processes) as pool:
        results = pool.map(run_single_ga_worker, process_args)
    
    end_time = perf_counter()
    total_runtime = end_time - start_time
    
    print("\n" + "="*60)
    print(f"All processes completed in {total_runtime:.2f}s")
    
    # Find overall best result
    best_result = max(results, key=lambda x: x['best_score'])
    print(f"Overall best score: {best_result['best_score']} (Process {best_result['process_id']})")
    
    # Save results if requested
    file_paths = {}
    if save_all_data:
        file_paths = save_results_to_files(results, run_dir, config_name, save_all_data)
    
    return results, total_runtime, file_paths

def test_parallel_ga_fixed():
    """Test the FIXED multiprocessing implementation."""
    print("🔧 Testing FIXED parallel GA implementation...")
    print("This version should work in Jupyter notebooks!")
    
    # Use small config for testing
    results, total_runtime, file_paths = run_parallel_ga_fixed(
        config_name="small", 
        num_processes=4, 
        num_generations=5,
        tmax=1000,  # Shorter simulation for testing
        save_all_data=True
    )
    
    # Analyze results
    if results:
        best_scores = [r['best_score'] for r in results]
        print(f"\n✅ SUCCESS! All processes completed")
        print(f"📊 Results:")
        print(f"   Best score: {max(best_scores)}")
        print(f"   Mean score: {np.mean(best_scores):.2f}")
        print(f"   Processes: {len(results)}")
        print(f"   Total runtime: {total_runtime:.2f}s")
        
        if file_paths:
            print(f"\n📁 Files saved:")
            for format_type, path in file_paths.items():
                print(f"   {format_type}: {path}")
    
    return results, total_runtime, file_paths


In [43]:
# TEST THE FIXED VERSION
print("🚀 Testing the fixed multiprocessing implementation!")
print("This should work in Jupyter notebooks on macOS.")
print()

# Run the fixed test
results, runtime, files = test_parallel_ga_fixed()


🚀 Testing the fixed multiprocessing implementation!
This should work in Jupyter notebooks on macOS.

🔧 Testing FIXED parallel GA implementation...
This version should work in Jupyter notebooks!
ℹ️  Using multiprocessing method: spawn
Starting 4 parallel GA processes
Config: small, Generations: 5, Population size: 100
Total individuals to evaluate: 2000
Results will be saved to: results/parallel_ga_small_20250726_210242
------------------------------------------------------------


Process SpawnPoolWorker-20:
Process SpawnPoolWorker-18:
Process SpawnPoolWorker-19:
Process SpawnPoolWorker-17:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/local/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/usr/local/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/queues.py", line 389, in get
    return _ForkingPickler.loads(res)
           ^^

KeyboardInterrupt: 

In [ ]:
import multiprocessing as mp
from multiprocessing import Pool
import os
from time import perf_counter
from datetime import datetime

def run_single_ga_instance(args):
    """
    Run a single genetic algorithm instance.
    This function will be executed by each process.
    
    Args:
        args: tuple containing (process_id, config_name, num_generations, random_seed)
    
    Returns:
        dict: Results from the GA run including best individual and scores
    """
    process_id, config_name, num_generations, random_seed, tmax = args
    
    # Set random seed for this process to ensure different initial populations
    np.random.seed(random_seed)
    
    # Print process start info
    print(f"Process {process_id}: Starting GA with seed {random_seed}")
    
    # Initialize population for this process
    cfg = GA_CONFIG[config_name]
    pop = initialize_population(cfg["POP_SIZE"], 
                              cfg["DNA_BOUNDS"][1], 
                              ACTIVE_SYNAPSES,
                              INHIBITORY_NEURONS)
    
    conn_map = initialize_connection_mapping(ACTIVE_SYNAPSES, NEURON_NAMES)
    
    # Track the best individual across generations
    best_individual = None
    best_score = -1
    generation_scores = []
    
    start_time = perf_counter()
    
    # Run the genetic algorithm
    for gen in range(num_generations):
        # Evaluate population
        scores = evaluate_population(pop,
                                   conn_map, N,                                    
                                   a, b, vreset, d, k, vr, vt, vpeak, C, E,
                                   alpha, cue_wave, go_wave,
                                   crit_Exp, crit_Cont, crit_indices, pass_ids,
                                   tmax, False)
        
        # Create population records
        dpops = [{"dna": pop[i], "dna_score": exp+cont} for i, (idx, exp, cont) in enumerate(scores)]
        
        # Track best individual
        current_best = max(dpops, key=lambda x: x["dna_score"])
        if current_best["dna_score"] > best_score:
            best_score = current_best["dna_score"]
            best_individual = current_best["dna"].copy()
        
        # Store generation statistics
        scores_list = [record["dna_score"] for record in dpops]
        generation_scores.append({
            'generation': gen,
            'best': max(scores_list),
            'mean': np.mean(scores_list),
            'std': np.std(scores_list)
        })
        
        # Print progress for this process
        if gen % 10 == 0 or gen == num_generations - 1:
            print(f"Process {process_id}: Gen {gen}, Best: {max(scores_list)}, Mean: {np.mean(scores_list):.1f}")
        
        # Generate next population (except for last generation)
        if gen < num_generations - 1:
            pop = spawn_next_population(dpops, cfg)
    
    end_time = perf_counter()
    runtime = end_time - start_time
    
    print(f"Process {process_id}: Completed in {runtime:.2f}s, Final best score: {best_score}")
    
    return {
        'process_id': process_id,
        'random_seed': random_seed,
        'best_individual': best_individual,
        'best_score': best_score,
        'generation_scores': generation_scores,
        'runtime': runtime,
        'final_population_size': len(pop)
    }


In [ ]:
def run_parallel_ga(config_name="F", num_processes=None, num_generations=None, tmax=5000):
    """
    Run multiple GA instances in parallel using multiprocessing.
    
    Args:
        config_name: Configuration to use from GA_CONFIG
        num_processes: Number of parallel processes (default: CPU count)
        num_generations: Override number of generations (default: from config)
        tmax: Maximum time steps for simulation
    
    Returns:
        list: Results from all GA instances
    """
    if num_processes is None:
        num_processes = mp.cpu_count()
    
    cfg = GA_CONFIG[config_name]
    if num_generations is None:
        num_generations = cfg["NUM_GENERATIONS"]
    
    print(f"Starting {num_processes} parallel GA processes")
    print(f"Config: {config_name}, Generations: {num_generations}, Population size: {cfg['POP_SIZE']}")
    print(f"Total individuals to evaluate: {num_processes * cfg['POP_SIZE'] * num_generations}")
    
    # Create arguments for each process with different random seeds
    base_seed = int(datetime.now().timestamp())
    process_args = []
    for i in range(num_processes):
        seed = base_seed + i * 1000  # Ensure seeds are well separated
        process_args.append((i, config_name, num_generations, seed, tmax))
    
    start_time = perf_counter()
    
    # Run processes in parallel
    with Pool(processes=num_processes) as pool:
        results = pool.map(run_single_ga_instance, process_args)
    
    end_time = perf_counter()
    total_runtime = end_time - start_time
    
    print(f"\nAll processes completed in {total_runtime:.2f}s")
    
    # Find overall best result
    best_result = max(results, key=lambda x: x['best_score'])
    print(f"Overall best score: {best_result['best_score']} (Process {best_result['process_id']})")
    
    return results, total_runtime


In [ ]:
def analyze_parallel_results(results, total_runtime):
    """
    Analyze and display results from parallel GA runs.
    """
    print("\n" + "="*80)
    print("PARALLEL GA RESULTS ANALYSIS")
    print("="*80)
    
    # Overall statistics
    best_scores = [r['best_score'] for r in results]
    runtimes = [r['runtime'] for r in results]
    
    print(f"Number of processes: {len(results)}")
    print(f"Total runtime: {total_runtime:.2f}s")
    print(f"Average process runtime: {np.mean(runtimes):.2f}s")
    print(f"Process runtime std: {np.std(runtimes):.2f}s")
    
    print(f"\nBest scores across processes:")
    print(f"  Best: {max(best_scores)}")
    print(f"  Mean: {np.mean(best_scores):.2f}")
    print(f"  Std:  {np.std(best_scores):.2f}")
    print(f"  Min:  {min(best_scores)}")
    
    # Per-process summary
    print(f"\nPer-process results:")
    print("Process | Best Score | Runtime(s) | Seed")
    print("-" * 45)
    for r in sorted(results, key=lambda x: x['best_score'], reverse=True):
        print(f"   {r['process_id']:2d}   |    {r['best_score']:4d}    |   {r['runtime']:6.2f}   | {r['random_seed']}")
    
    # Find the overall best individual
    best_result = max(results, key=lambda x: x['best_score'])
    print(f"\nBest individual found:")
    print(f"  Process: {best_result['process_id']}")
    print(f"  Score: {best_result['best_score']}")
    print(f"  DNA (first 10 genes): {best_result['best_individual'][:10]}")
    
    return best_result

def test_parallel_ga():
    """Test the multiprocessing implementation with a small configuration."""
    print("Testing parallel GA with small configuration...")
    
    # Use small config for testing
    results, total_runtime = run_parallel_ga(
        config_name="small", 
        num_processes=4, 
        num_generations=5,
        tmax=1000  # Shorter simulation for testing
    )
    
    best_result = analyze_parallel_results(results, total_runtime)
    return results, best_result


In [ ]:
# Test the multiprocessing implementation
if __name__ == '__main__':
    # First, let's test with a small configuration
    print("Testing multiprocessing GA implementation...")
    results, best_result = test_parallel_ga()


In [28]:
# Example: Run multiprocessing GA with your original configuration
print("Running parallel GA with config 'F' using 4 processes...")
print("This will run 4 independent GA instances in parallel, each with different initial populations.")

# Run with a smaller number of generations for demonstration
results, runtime = run_parallel_ga(
    config_name="F", 
    num_processes=4, 
    num_generations=10,  # Reduced for demo
    tmax=5000
)

# Analyze the results
best_result = analyze_parallel_results(results, runtime)

print("\nComparison:")
print(f"Your original single-process run found scores up to ~938")
print(f"This parallel run found a best score of: {best_result['best_score']}")
print(f"With {len(results)} processes, we explored {len(results)} different starting populations!")


Running parallel GA with config 'F' using 4 processes...
This will run 4 independent GA instances in parallel, each with different initial populations.


NameError: name 'run_parallel_ga' is not defined